# How-to use JAX with `qiskit-dynamics`


JAX enables just-in-time compilation, automatic differentation, and GPU execution. JAX is integrated into `qiskit-dynamics` via the `dispatch` submodule, which allows most parts of the package to be executed with either `numpy` or `jax.numpy`. 

This guide demonstrates how to write JAX-transformable code in `qiskit-dynamics`:

1. How-to set JAX as the default numerical backend.
2. Writing code that can be executed by both `numpy` or `jax.numpy` using the `dispatch.Array` class.
3. Writing JAX-transformable functions in `qiskit-dynamics`.

## 1. Setting JAX as the default backend

The `dispatch` submodule provides a means of controlling whether array operations are performed using `numpy` or `jax.numpy`. In many cases, the "default backend" is used to determine which of the two options is used.

In [1]:
# configure jax to use 64 bit mode
import jax
jax.config.update("jax_enable_x64", True)

# import dispatch and set default backend
from qiskit_dynamics import dispatch
dispatch.set_default_backend('jax')

The default backend can be observed via:

In [2]:
dispatch.default_backend()

'jax'

## 2. Writing code that can be executed by both `numpy` and `jax.numpy` using the `dispatch.Array` class

The `dispatch.Array` class wraps both `numpy` and `jax.numpy` arrays. The particular type is indicated by the `backend` property, and `numpy` functions called on an `Array` will automatically be dispatched to `numpy` or `jax.numpy` based on the `Array`'s backend.

To write functions that can be executed by both `numpy` or `jax`:
- First wrap the array in an `Array` of the right type.
- Call `numpy` functions on the `Array`.
- If necessary, extract the underlying raw array via the `.data` property.

In [3]:
import numpy as np

from qiskit_dynamics.dispatch import Array

We will write a function that takes the `sine` of the values of an array, using either JAX or numpy.

In [4]:
def dispatched_sin(x):
    # wrap in an Array
    x = Array(x)
    return np.sin(x).data

We have already set JAX as the default backend, so the above `Array` wrapping will automatically set the backend to `'jax'`, and the call to `np.sin` will be dispatched to `jax.numpy.sin`:

In [5]:
output = dispatched_sin(np.array([1., 2., 3.]))
output

DeviceArray([0.84147098, 0.90929743, 0.14112001], dtype=float64)

The output of the above is a `DeviceArray`, which is the jax analogy of `numpy.ndarray`. 

We also explicitly pass an `Array` with `numpy` backend to the function, and it will be executed with `numpy`.

In [6]:
output2 = dispatched_sin(Array([1., 2., 3.], backend='numpy'))
output2

array([0.84147098, 0.90929743, 0.14112001])

## 3. Writing JAX-transformable functions in `qiskit-dynamics`

JAX-transformable functions must be:
- JAX-executable.
- Take as JAX arrays as input and output (see the [JAX documentation](https://jax.readthedocs.io/en/latest/) for more details on accepted input and output types).
- Pure, in the sense that they have no side-effects.

The previous section shows how to handle the first two points using `dispatch` and `Array`. The last point further restricts the type of code that can be safely transformed. 

`qiskit-dynamics` uses various objects which can be updated by setting properties (models, solvers). If a function to be transformed requires updating an already-constructed object of this form, it is necessary to first make a *copy*. We demonstrate this here with an anticipated common use-case: parameterized simulation of a model of a quantum system.

We outline:
- Defining a `Solver` with a model of a qubit, without specifying the signal coefficients.
- Writing a function that simulates driving the system with a piecewise constant envelope.
   - The input parameters are the piecewise constant envelope values.
   - The output is the final unitary.
   - Before updating the signals in the function, we *copy* the `Solver`.
- We JAX compile and run this function.

In [7]:
from qiskit.quantum_info import Operator
from qiskit_dynamics import Solver
from qiskit_dynamics.signals import DiscreteSignal

First, instantiate the `Solver` object *without* signals. These will be inserted in the function to be transformed.

In [8]:
drift = 2 * np.pi * 5 * Operator.from_label('Z') / 2
operators = [2 * np.pi * 0.02 * Operator.from_label('X') / 2]

ham_solver = Solver(hamiltonian_operators=operators,
                    drift=drift,
                    rotating_frame=drift)

Next, define the function.

In [9]:
def simulation_function(samples):
    
    # define signal
    signal = DiscreteSignal(samples=samples, dt=1., carrier_freq=5.)
    
    # make copy and set signal
    solver_copy = ham_solver.copy()
    solver_copy.signals = [signal]
    
    # call solve with a jax-compatible method
    results = solver_copy.solve(t_span=[0, 1. * len(samples)], y0=np.eye(2, dtype=complex), 
                               method='jax_odeint', atol=1e-8, rtol=1e-8)
    
    # return raw array of final results
    return results.y[-1]

We can now `jit` this function.

In [10]:
from jax import jit
jit_simulation_function = jit(simulation_function)

The first time the function is called, it will compile and then execute. Hence, the time taken  on the first call *includes* compilation time.

In [11]:
# block_until_ready is used to make sure time is correctly measured
%time jit_simulation_function(np.ones(100)).block_until_ready()

CPU times: user 626 ms, sys: 8.74 ms, total: 634 ms
Wall time: 632 ms


DeviceArray([[-1.00000012e+00+4.28459274e-11j,
               7.45354887e-09-3.79530561e-07j],
             [-7.45354888e-09-3.79530561e-07j,
              -1.00000012e+00-4.28459249e-11j]], dtype=complex128)

The second time it is called, the compiled function is executed, and therefore demonstrates the true time of the compiled function.

In [12]:
%time jit_simulation_function(2 * np.ones(100)).block_until_ready()

CPU times: user 15.2 ms, sys: 1.38 ms, total: 16.6 ms
Wall time: 14.7 ms


DeviceArray([[ 1.00000018e+00-3.19374866e-09j,
              -2.30122175e-09+3.18206246e-06j],
             [ 2.30122174e-09+3.18206246e-06j,
               1.00000018e+00+3.19374866e-09j]], dtype=complex128)